# interrupt 승인 게이트

로컬 쓰기 전에 승인 여부를 확인해야 거절된 작업이 실행되지 않는지 확인합니다.


## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.


In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11

Note: you may need to restart the kernel to use updated packages.


## 승인 대상과 기록 파일 준비

되돌릴 수 없는 동작을 대신해 로컬 파일 기록을 사용합니다.

`interrupt(payload)`는 노드 실행을 멈추고 payload를 호출자에게 반환합니다. 이를 사용하려면 상태를 저장할 체크포인터와 재개할 실행을 찾는 `thread_id`가 필요합니다. 첫 `invoke`의 결과에는 `__interrupt__`가 들어 있고, `Command(resume=value)`로 같은 스레드를 호출하면 그 값이 노드 안 `interrupt()`의 반환값이 됩니다.

재개는 interrupt 다음 줄에서 프로세스를 그대로 계속하는 방식이 아닙니다. 런타임은 노드를 처음부터 다시 실행하며 저장된 재개 값을 같은 interrupt 호출에 전달합니다. 따라서 interrupt 앞의 `write_effect`는 첫 실행과 재개에서 두 번 실행될 수 있습니다. 승인 게이트는 부작용보다 앞에 두어야 거절이 실제 실행을 막습니다.


In [2]:
# 승인 결과를 기록할 로컬 파일과 HITL 상태 형식을 준비합니다.
from pathlib import Path
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

candidates = [Path.cwd(), *Path.cwd().parents]
lab = next((p / "labs/part6/ch24-human-in-the-loop" for p in candidates if (p / "labs/part6/ch24-human-in-the-loop").is_dir()), None)
if lab is None:
    lab = next(p for p in candidates if p.name == "ch24-human-in-the-loop")
effect_path = lab / "approval-effects.txt"
effect_path.unlink(missing_ok=True)

class State(TypedDict):
    request: str
    status: str

def write_effect(request: str) -> None:
    with effect_path.open("a", encoding="utf-8") as stream:
        stream.write(request + "\n")

## Before: 부작용 뒤에 승인 요청


In [3]:
# 부작용 뒤 interrupt를 둔 그래프를 거절해도 파일 기록이 남는지 확인합니다.
def unsafe_node(state: State) -> dict:
    write_effect(state["request"])
    approved = interrupt({"request": state["request"], "question": "승인합니까?"})
    return {"status": "approved" if approved else "rejected"}

unsafe_builder = StateGraph(State)
unsafe_builder.add_node("gate", unsafe_node)
unsafe_builder.add_edge(START, "gate")
unsafe_builder.add_edge("gate", END)
unsafe = unsafe_builder.compile(checkpointer=InMemorySaver())
unsafe_config = {"configurable": {"thread_id": "unsafe"}}
paused = unsafe.invoke({"request": "transfer-local", "status": "new"}, unsafe_config)
rejected = unsafe.invoke(Command(resume=False), unsafe_config)
print("status:", rejected["status"])
print("writes despite rejection:", effect_path.read_text(encoding="utf-8").splitlines())

status: rejected
writes despite rejection: ['transfer-local', 'transfer-local']


## After: 승인 뒤에만 부작용 실행

파일 쓰기는 송금, 배포, 삭제처럼 되돌리기 어려운 동작을 대신한 관찰 가능한 예입니다. 읽기 전용 검색까지 매번 승인받으면 흐름만 느려질 수 있으므로 위험한 도구 이름이나 인자 조건에만 게이트를 두는 편이 적절합니다.

승인 화면을 보여 줬다는 사실만으로 안전하지 않습니다. 외부 동작이 승인 전에 실행됐는지, 재개 때 다시 실행되는지 코드 순서로 확인해야 합니다. safe 경로의 파일 부재는 False가 외부 쓰기 전에 검사됐다는 이번 실습의 증거입니다.


In [4]:
# interrupt를 먼저 둔 그래프를 거절해 파일 자체가 생성되지 않는지 확인합니다.
effect_path.unlink(missing_ok=True)
def safe_node(state: State) -> dict:
    approved = interrupt({"request": state["request"], "question": "승인합니까?"})
    if not approved:
        return {"status": "rejected"}
    write_effect(state["request"])
    return {"status": "approved"}

safe_builder = StateGraph(State)
safe_builder.add_node("gate", safe_node)
safe_builder.add_edge(START, "gate")
safe_builder.add_edge("gate", END)
safe = safe_builder.compile(checkpointer=InMemorySaver())
safe_config = {"configurable": {"thread_id": "safe"}}
safe.invoke({"request": "transfer-local", "status": "new"}, safe_config)
rejected = safe.invoke(Command(resume=False), safe_config)
print("status:", rejected["status"])
print("effect file exists:", effect_path.exists())
assert rejected["status"] == "rejected" and not effect_path.exists()

status: rejected
effect file exists: False
